<a href="https://colab.research.google.com/github/Abraham1439/Rag_Reglamento_Biblioteca/blob/main/Rag_Reglamento_Biblioteca_Duoc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Asistente Virtual RAG de Biblioteca Duoc UC

Este notebook reúne todo el código del pipeline RAG del proyecto **Biblioteca Duoc UC** en un solo archivo ejecutable directamente en **Google Colab**.

### Arquitectura del Notebook:
1. **Instalación de Dependencias**: `openai`, `langchain`, `faiss-cpu`, etc.
2. **Configuración y Claves API**: Groq (LLM) y Mistral (Embeddings).
3. **Carga y Fragmentación (Ingesta)**: Creación de documentos del reglamento y chunking.
4. **Índice Vectorial (FAISS)**: Generación de embeddings e indexación.
5. **Prompt Engineering**: Estrategias Zero-Shot, Few-Shot y Chain-of-Thought.
6. **Agente RAG + Filtro de Confianza**: Lógica de respuesta y derivación.
7. **Evaluación de Casos**: Pruebas con consultas dentro y fuera de alcance.
8. **Comparativa de Técnicas de Prompting**: Benchmark empírico de Zero-Shot, Few-Shot y Chain-of-Thought.

## 1. Instalación de Dependencias

In [ ]:
!pip install -q openai langchain langchain-core langchain-openai langchain-community langchain-text-splitters faiss-cpu python-dotenv numpy

## 2. Configuración del Entorno y Claves API

In [ ]:
import os
import getpass

# --- Cargar Credenciales (Google Colab Secrets o .env Local) ---
try:
    from google.colab import userdata          # Entra aquí si estás en Google Colab

    # Lista de variables a buscar en los Secrets de Colab
    colab_keys = (
        "LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL",
        "EMBEDDING_API_KEY", "EMBEDDING_BASE_URL", "EMBEDDING_MODEL"
    )
    for key in colab_keys:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass                                # Si el Secret no existe en Colab, no hace nada
except ImportError:
    from dotenv import load_dotenv             # Entra aquí si estás corriendo en tu máquina local
    load_dotenv()

# --- Configuración por defecto si no están definidas en entorno/secrets ---
os.environ.setdefault("LLM_BASE_URL", "https://api.groq.com/openai/v1")
os.environ.setdefault("LLM_MODEL", "groq/compound-mini")

os.environ.setdefault("EMBEDDING_BASE_URL", "https://api.mistral.ai/v1")
os.environ.setdefault("EMBEDDING_MODEL", "mistral-embed")

# --- Asignación de Variables ---
LLM_BASE_URL = os.getenv("LLM_BASE_URL")
LLM_MODEL = os.getenv("LLM_MODEL")
# Busca en Secrets/env; si no existe, te pedirá ingresarla interactivamente como respaldo
LLM_API_KEY = os.getenv("LLM_API_KEY") or getpass.getpass("Ingresa tu LLM_API_KEY (Groq): ")

EMBEDDING_BASE_URL = os.getenv("EMBEDDING_BASE_URL")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
EMBEDDING_API_KEY = os.getenv("EMBEDDING_API_KEY") or getpass.getpass("Ingresa tu EMBEDDING_API_KEY (Mistral): ")

# --- Configuración del Proyecto ---
from pathlib import Path

def localizar_data(inicio=None):
    inicio = Path.cwd() if inicio is None else Path(inicio)
    for candidato in (inicio.resolve(), *inicio.resolve().parents):
        if (candidato / "notebook" / "asistente_biblioteca_duoc.ipynb").is_file():
            carpeta = candidato / "data"
            if not carpeta.is_dir():
                raise FileNotFoundError(f"Falta la carpeta de datos del proyecto: {carpeta}")
            return carpeta
    raise FileNotFoundError(
        "Abre el notebook desde la carpeta del proyecto o desde su subcarpeta notebook. "
        "En Colab, carga el proyecto completo conservando data/ y notebook/."
    )

DATA_DIR = str(localizar_data())
CHUNK_SIZE = 500
CHUNK_OVERLAP = 80
TOP_K = 3
CONFIDENCE_THRESHOLD = 0.45

print("Configuración cargada correctamente:")
print(f"- LLM Model: {LLM_MODEL} (Groq)")
print(f"- Embedding Model: {EMBEDDING_MODEL} (Mistral)")